In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-09-01-replay-phenotypes"


# get data https://osf.io/ugzyd


In [ ]:
truebb_cigar = 29
truebb_other = 40
selfbb_cigar = 51
selfbb_other = 18

n_true = truebb_cigar + truebb_other
n_self = selfbb_cigar + selfbb_other
p_true = truebb_cigar / n_true
p_self = selfbb_cigar / n_self

common_n = 69
assert common_n == n_true == n_self


# statistical analysis


In [ ]:
true_lo, true_hi = scipy_stats.binomtest(truebb_cigar, n_true).proportion_ci(
    confidence_level=0.95, method="exact"
)
self_lo, self_hi = scipy_stats.binomtest(selfbb_cigar, n_self).proportion_ci(
    confidence_level=0.95, method="exact"
)

print(f"truebb: {truebb_cigar}/{n_true} = {100 * p_true:.1f}%")
print(f"  Clopper-Pearson 95% CI: {true_lo:.4f} to {true_hi:.4f}")
print(
    f"  Out of {common_n}: {p_true * common_n:.1f} "
    f"({true_lo * common_n:.1f} to {true_hi * common_n:.1f})"
)
print(f"selfbb: {selfbb_cigar}/{n_self} = {100 * p_self:.1f}%")
print(f"  Clopper-Pearson 95% CI: {self_lo:.4f} to {self_hi:.4f}")
print(
    f"  Out of {common_n}: {p_self * common_n:.1f} "
    f"({self_lo * common_n:.1f} to {self_hi * common_n:.1f})"
)

rr_self_vs_true = p_self / p_true
rr_true_vs_self = p_true / p_self
print(f"\nRisk ratio (selfbb vs truebb): {rr_self_vs_true:.4f}")
print(f"Risk ratio (truebb vs selfbb): {rr_true_vs_self:.4f}")

table = np.array([[selfbb_cigar, selfbb_other], [truebb_cigar, truebb_other]])
odds_ratio, p_value = scipy_stats.fisher_exact(table, alternative="two-sided")
print(f"\nFisher's exact: OR = {odds_ratio:.4f}, two-sided p = {p_value:.6g}")


In [ ]:
stats_by_group = {
    "truebb": (true_lo * n_true, p_true * n_true, true_hi * n_true),
    "selfbb": (self_lo * n_self, p_self * n_self, self_hi * n_self),
}

df = pd.DataFrame(
    [
        {"source": group, "prop": value}
        for group, values in stats_by_group.items()
        for value in values
    ]
)

with tp.teed(
    sns.pointplot,
    data=df,
    x=None,
    y="prop",
    errorbar=None,
    legend=False,
    hue="source",
    dodge=False,
    hue_order=["truebb", "selfbb"],
    palette=["white", "white"],
    estimator=np.median,
    markersize=2,
    teeplot_subdir=teeplot_subdir,
) as ax:
    sns.pointplot(
        ax=ax,
        data=df,
        x=None,
        y="prop",
        legend=False,
        hue="source",
        dodge=False,
        hue_order=["selfbb", "truebb"],
        palette=[sns.color_palette("Dark2")[0], "#a3cf73"],
        estimator=np.median,
        errorbar=lambda v: (v.min(), v.max()),
        linewidth=2.5,
        markersize=4,
        zorder=-1,
    )
    ax.set_ylim(0, common_n)
    ax.set_ylabel("N Cigar After Replay")
    ax.set_yticks([0, selfbb_cigar, truebb_cigar, common_n])
    ax.set_xticks([])
    ax.figure.set_size_inches(0.5,2)
    sns.despine(ax=ax)

    bracket_x = 0.4
    tick = 0.1
    y0, y1 = selfbb_cigar, truebb_cigar
    p_text = "p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"

    ax.plot(
        [bracket_x - tick, bracket_x, bracket_x, bracket_x - tick],
        [y0, y0, y1, y1],
        lw=1.0,
        color="0.2",
        clip_on=False,
        solid_capstyle="butt",
    )
    ax.text(
        bracket_x + 2.5 * tick,
        (y0 + y1) / 2,
        f"RR$={rr_self_vs_true:.2f}$, ${p_text}$",
        rotation=90,
        ha="left",
        va="center",
        fontsize=9,
        clip_on=False,
    )
    ax.set_xlim(-0.5, 1)

    labels = {"selfbb": "self", "truebb": "coevo"}
    colors = {"selfbb": sns.color_palette("Dark2")[0], "truebb": sns.color_palette("Dark2")[4]}
    pad = 2

    order = sorted(stats_by_group, key=lambda g: stats_by_group[g][1])
    low_group, high_group = order[0], order[-1]

    ax.text(
        0,
        stats_by_group[high_group][2] + pad,
        labels[high_group],
        rotation=90,
        ha="center",
        va="bottom",
        color=colors[high_group],
        fontsize=9,
        clip_on=False,
    )
    ax.text(
        0,
        stats_by_group[low_group][0] - pad,
        labels[low_group],
        rotation=90,
        ha="center",
        va="top",
        color=colors[low_group],
        fontsize=9,
        clip_on=False,
    )
